# 03 - Previsione GFS e formato GRIB

Nei notebook precedenti abbiamo lavorato con osservazioni: dati misurati, gia' accaduti.
Qui entra in scena la previsione: l'uscita di un modello numerico, il GFS (Global Forecast
System) della NOAA, distribuito in formato GRIB2. E' il primo notebook del percorso che
scarica un file binario invece di CSV o Parquet, e il primo che affronta il problema
concettuale centrale del post-processing: come si porta un valore di griglia su un punto
stazione.


## Passo 1 - Prerequisiti e ricostruzione dell'area dalla staffetta

Il notebook 02 ha scelto una regione e scritto `01_stations.csv`. Questo notebook non
sceglie una propria area: la eredita da li', leggendo la colonna `regione_scelta`. E'
la stessa staffetta che collegherà i notebook successivi: ogni fase eredita il contesto
della precedente invece di ridefinirlo.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import common
import pandas as pd

common.richiede("01_stations.csv")
stazioni = pd.read_csv(common.data_path("01_stations.csv"))
REGIONE = stazioni["regione_scelta"].iloc[0]
lon_min, lon_max, lat_min, lat_max = common.get_bbox(REGIONE)
print(f"Area ereditata dal notebook 02: {REGIONE} ({len(stazioni)} stazioni)")


## Passo 2 - Il filtro NOMADS: scaricare megabyte, non gigabyte

Un run completo di GFS a 0.25 gradi contiene centinaia di variabili su decine di livelli,
per ogni ora di previsione: diversi gigabyte. NOMADS espone pero' un servizio di filtro
(`filter_gfs_0p25_1hr.pl`) che ritaglia, sul lato server, sia le variabili sia l'area
geografica prima di inviare la risposta. Chiedendo solo la temperatura a 2 metri (T2m)
sul bbox della regione scelta, il download scende a poche centinaia di byte per lead
time. E' la stessa procedura descritta come Misura 2 nel documento di fattibilita' dei
dati (`docs/05-mvp-data-feasibility.md`): non serve l'archivio intero per validare il
metodo.

Si scelgono piu' "lead time" (ore di proiezione dal run: +3h, +6h, ... +48h) per avere,
piu' avanti, una previsione che si allunga nel tempo invece di un singolo istante.


In [ ]:
from datetime import datetime, timedelta, timezone

# Si sceglie un run recente ma gia' pubblicato: NOMADS impiega qualche ora
# a rendere disponibile un ciclo completo.
adesso = datetime.now(timezone.utc) - timedelta(hours=8)
run = adesso.replace(hour=(adesso.hour // 6) * 6, minute=0, second=0, microsecond=0)
LEADS = [3, 6, 9, 12, 24, 36, 48]

BASE = "https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25_1hr.pl"
percorsi = {}
for lead in LEADS:
    params = {
        "file": f"gfs.t{run:%H}z.pgrb2.0p25.f{lead:03d}",
        "lev_2_m_above_ground": "on",
        "var_TMP": "on",
        "subregion": "",
        "leftlon": lon_min, "rightlon": lon_max,
        "toplat": lat_max, "bottomlat": lat_min,
        "dir": f"/gfs.{run:%Y%m%d}/{run:%H}/atmos",
    }
    dest = common.RAW_DIR / f"gfs_{run:%Y%m%d_%H}_f{lead:03d}.grib2"
    try:
        p = common.scarica_con_cache(BASE, dest, params=params)
        if p.read_bytes()[:4] != b"GRIB":
            print(f"  lead {lead:3d}h: risposta non GRIB, run forse non ancora pubblicato")
            p.unlink(missing_ok=True)
            continue
        percorsi[lead] = p
        print(f"  lead {lead:3d}h: {p.stat().st_size/1024:.1f} KB")
    except Exception as e:
        print(f"  lead {lead:3d}h: fallito ({e})")

print(f"\nRun {run:%Y-%m-%d %H} UTC, {len(percorsi)} lead scaricati")


Se la cella sopra riporta **zero lead scaricati**, il run scelto e' troppo recente:
NOMADS non lo ha ancora pubblicato per intero. Il rimedio e' aumentare l'offset nella
riga `adesso = datetime.now(timezone.utc) - timedelta(hours=8)`, per esempio a 14 o a 20
ore, e rieseguire la cella. Piu' l'offset e' grande, piu' e' probabile che il run sia
gia' stabile su NOMADS.


## Passo 3 - Aprire il GRIB

Un file GRIB e' un contenitore di messaggi: ogni messaggio e' una singola combinazione
di variabile, livello verticale e istante temporale. Non e' una tabella, e' un formato
binario compresso pensato per l'interscambio fra centri meteorologici. `xarray`, con il
motore `cfgrib` (basato sulla libreria ECMWF `eccodes`), lo presenta come un array
multidimensionale con coordinate esplicite: `time` e' l'istante di emissione del run,
`step` e' il lead time (la distanza dal run), e `valid_time` e' la loro somma, cioe'
l'istante a cui la previsione si riferisce davvero.


In [ ]:
import xarray as xr

lead_demo = sorted(percorsi)[0]
ds = xr.open_dataset(percorsi[lead_demo], engine="cfgrib")
print(ds)
print("\nRun (time):      ", pd.Timestamp(ds.time.values, tz="UTC"))
print("Lead (step):     ", pd.Timedelta(ds.step.values))
print("Valid time:      ", pd.Timestamp(ds.valid_time.values, tz="UTC"))
print("Forma della griglia:", ds.t2m.shape)
print("Passo in gradi:  ",
      float(abs(ds.latitude[1] - ds.latitude[0])),
      float(abs(ds.longitude[1] - ds.longitude[0])))


## Passo 4 - Prima mappa vera del percorso: Italia intera, poi zoom

Questa e' la prima mappa del percorso che disegna un campo di modello invece di punti
osservati. Per non perdere la percezione di cosa sia una griglia 0.25 gradi, la si
guarda prima sull'Italia intera - dove la risoluzione della griglia rispetto alla
scala del paese e' evidente - e poi in zoom sulla sola regione scelta, dove i valori
diventano confrontabili con le stazioni.


In [ ]:
import matplotlib.pyplot as plt
try:
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    proj = ccrs.PlateCarree()
except Exception:
    ccrs = None

t2m_c = ds.t2m - 273.15  # da kelvin a gradi Celsius

fig, assi = plt.subplots(1, 2, figsize=(15, 6),
                         subplot_kw={"projection": proj} if ccrs else None)

for ax, estensione, titolo in [
    (assi[0], common.BBOX_ITALIA, "Inquadramento: Italia"),
    (assi[1], (lon_min, lon_max, lat_min, lat_max), f"Zoom: {REGIONE}"),
]:
    kw = {"transform": proj} if ccrs else {}
    if ccrs:
        ax.set_extent(estensione, crs=proj)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":")
    else:
        ax.set_xlim(estensione[0], estensione[1]); ax.set_ylim(estensione[2], estensione[3])
    m = ax.pcolormesh(ds.longitude, ds.latitude, t2m_c, shading="nearest",
                      cmap="RdYlBu_r", **kw)
    ax.scatter(stazioni["lon"], stazioni["lat"], s=60, c="black", marker="^",
               zorder=5, label="stazioni", **kw)
    ax.set_title(titolo)
    ax.legend(loc="lower left", fontsize=8)
    plt.colorbar(m, ax=ax, label="T2m (C)", shrink=0.8)

plt.suptitle(f"GFS run {run:%Y-%m-%d %H} UTC, lead +{lead_demo}h")
plt.tight_layout(); plt.show()


## Passo 5 - Dalla griglia al punto

Questo e' il passaggio concettuale centrale del percorso.

Una stazione e' un punto con una latitudine e una longitudine esatte. Il modello, invece,
ha valori solo sui nodi della sua griglia regolare: fra un nodo e l'altro non esiste
nessun valore calcolato. Per confrontare previsione e osservazione nello stesso punto
serve quindi interpolare.

- **Nearest** (vicino piu' prossimo): prende il valore del nodo di griglia piu' vicino
  alla stazione. Non inventa nulla - il valore restituito e' sempre uno di quelli che
  il modello ha davvero calcolato - ma introduce discontinuita' brusche: due stazioni
  vicine ma su lati opposti di un confine di cella possono ricevere valori identici o
  bruscamente diversi a seconda di dove cade quel confine.
- **Bilineare**: media pesata dei quattro nodi che circondano il punto, pesata per
  distanza. Il risultato e' piu' liscio e realistico come andamento spaziale, ma puo'
  produrre un valore che nessun singolo nodo della griglia aveva davvero.

La differenza fra i due metodi, pero', e' un dettaglio rispetto a un'altra fonte di
errore, molto piu' importante: **la quota**. Il modello non conosce l'orografia reale,
conosce un'orografia mediata sull'area di ciascuna cella di griglia. Una stazione posta
a 3.488 metri, dentro una cella la cui quota media (fra valli e crinali) e' molto piu'
bassa, ricevera' sistematicamente una temperatura sovrastimata - perche' il modello,
in quel punto, "pensa" di essere piu' in basso di quanto sia la stazione. Ed e'
esattamente la ragione fisica per cui il post-processing statistico funziona: quell'
errore non e' casuale, e' un bias ripetibile legato alla differenza di quota fra
modello e stazione, e un bias ripetibile e' un bias che si puo' imparare e correggere.


In [ ]:
PUBBLICAZIONE_STIMATA_ORE = 4  # ritardo tipico fra run e disponibilita' su NOMADS

righe = []
for lead, percorso in sorted(percorsi.items()):
    d = xr.open_dataset(percorso, engine="cfgrib")
    t_c = d.t2m - 273.15
    for _, s in stazioni.iterrows():
        nearest = float(t_c.sel(latitude=s["lat"], longitude=s["lon"], method="nearest"))
        bilin = float(t_c.interp(latitude=s["lat"], longitude=s["lon"]))
        righe.append({
            "station_id": s["station_id"],
            "run_time_utc": pd.Timestamp(d.time.values, tz="UTC"),
            "publication_time_utc": pd.Timestamp(d.time.values, tz="UTC")
                                    + pd.Timedelta(hours=PUBBLICAZIONE_STIMATA_ORE),
            "lead_hours": lead,
            "valid_time_utc": pd.Timestamp(d.valid_time.values, tz="UTC"),
            "t2m_c_forecast": bilin,
            "t2m_c_forecast_nearest": nearest,
            "station_elev_m": s["elev_m"],
            "metodo_interpolazione": "bilineare",
        })

fc = pd.DataFrame(righe)
print(fc.head(10).to_string(index=False))
print("\nDifferenza fra nearest e bilineare (C):")
print((fc["t2m_c_forecast"] - fc["t2m_c_forecast_nearest"]).describe().to_string())

fc.to_parquet(common.data_path("03_forecast_points.parquet"), index=False)
print("\nScritto", common.data_path("03_forecast_points.parquet"))


## Il limite di quello che hai fatto

- **NOMADS conserva circa dieci giorni.** Quello che hai scaricato non e' un archivio
  storico: fra un mese quel run non ci sara' piu'. Per il vero storico serve l'archivio
  S3 `noaa-gfs-bdp-pds`, mostrato sotto ma non eseguito perche' richiede tempo e spazio
  molto maggiori.
- `publication_time_utc` qui e' **stimato** con un ritardo fisso di 4 ore. Nella
  pipeline vera va registrato quando il file e' stato davvero visto disponibile,
  perche' e' il campo su cui si fonda tutta la regola anti-leakage: un modello non
  puo' mai essere valutato con un dato che, all'istante della previsione, non era
  ancora stato pubblicato.
- Hai una sola variabile (T2m) e un solo run: sufficiente per il metodo, non per una
  valutazione. Servono piu' run e piu' variabili per dire qualcosa sulla qualita'
  del modello.


In [ ]:
# Non eseguire ora: e' il modo di prendere lo storico vero.
#   aws s3 ls --no-sign-request s3://noaa-gfs-bdp-pds/gfs.20240115/00/atmos/
# Ordini di grandezza nel documento docs/05-mvp-data-feasibility.md.
